# PIMSR Overlay Smoke Test

把這個 notebook 和 `pimsr.bit`、`pimsr.hwh` 放在同一個資料夾，然後從上到下執行。這版會用目前硬體預設的 `4x4` LR 測試圖，跑完後比對 RTL golden 的 `8x8` HR 輸出。

In [ ]:
from pathlib import Path
import time
import numpy as np

print('Current directory:', Path.cwd())
print('Files:')
for p in sorted(Path.cwd().iterdir()):
    if p.suffix in {'.bit', '.hwh', '.py', '.ipynb'}:
        print(' ', p.name)

In [ ]:
from pynq import Overlay, MMIO, Device

print('PYNQ devices:', Device.devices)

In [ ]:
BIT_FILE = 'pimsr.bit'

CTRL_BASE = 0x43C00000
CTRL_SIZE = 0x10000
HR_BASE = 0x40000000
LR_BASE = 0x42000000
BRAM_SIZE = 0x2000

REG_CONTROL = 0x00
REG_WIDTH = 0x04
REG_HEIGHT = 0x08

DONE_MASK = 1 << 1
BUSY_MASK = 1 << 2

IMG_W = 4
IMG_H = 4
SCALE = 2

LR_4X4 = np.array([
    [0x00, 0x11, 0x22, 0x33],
    [0x29, 0x47, 0x65, 0x83],
    [0x52, 0x7D, 0xA8, 0xD3],
    [0x7B, 0xB3, 0xEB, 0x23],
], dtype=np.uint8)

HR_8X8_GOLDEN = np.array([
    [0x00, 0x00, 0x01, 0x1B, 0x1D, 0x21, 0x2E, 0x33],
    [0x01, 0x00, 0x02, 0x1C, 0x22, 0x25, 0x2F, 0x34],
    [0x09, 0x0C, 0x27, 0x3C, 0x47, 0x59, 0x3A, 0x3F],
    [0x40, 0x44, 0x55, 0x69, 0x73, 0x85, 0xB2, 0xB7],
    [0x47, 0x4C, 0x5F, 0x74, 0x73, 0x94, 0xF1, 0xF3],
    [0x4F, 0x55, 0x7E, 0x92, 0xA9, 0xCA, 0xC8, 0xCA],
    [0x72, 0x76, 0x79, 0xDC, 0xFF, 0xE8, 0x22, 0x02],
    [0x78, 0x7C, 0x7D, 0xE0, 0xFF, 0xEB, 0x12, 0x00],
], dtype=np.uint8)

In [ ]:
def pack_u8_to_u32_le(values):
    flat = np.asarray(values, dtype=np.uint8).reshape(-1)
    words = []
    for base in range(0, len(flat), 4):
        word = 0
        for lane, value in enumerate(flat[base:base + 4]):
            word |= int(value) << (8 * lane)
        words.append(word)
    return words


def unpack_u32_to_u8_le(words, nbytes):
    out = np.zeros(nbytes, dtype=np.uint8)
    idx = 0
    for word in words:
        for lane in range(4):
            if idx >= nbytes:
                break
            out[idx] = (int(word) >> (8 * lane)) & 0xFF
            idx += 1
    return out


def write_image_to_bram(mmio, image):
    for offset, word in enumerate(pack_u8_to_u32_le(image)):
        mmio.write(offset * 4, word)


def clear_bram(mmio, nbytes):
    for offset in range(0, nbytes, 4):
        mmio.write(offset, 0)


def read_image_from_bram(mmio, height, width):
    nbytes = height * width
    words = [mmio.read(offset) for offset in range(0, nbytes, 4)]
    return unpack_u32_to_u8_le(words, nbytes).reshape(height, width)


def wait_done(ctrl, timeout_s=1.0):
    deadline = time.monotonic() + timeout_s
    last_status = 0
    while time.monotonic() < deadline:
        last_status = ctrl.read(REG_CONTROL)
        if last_status & DONE_MASK:
            return last_status
        time.sleep(0.001)
    raise TimeoutError(
        f'PIMSR timeout: status=0x{last_status:08x}, '
        f'done={(last_status & DONE_MASK) != 0}, busy={(last_status & BUSY_MASK) != 0}'
    )

In [ ]:
bit_path = Path(BIT_FILE)
hwh_path = bit_path.with_suffix('.hwh')

assert bit_path.exists(), f'Missing {bit_path}'
assert hwh_path.exists(), f'Missing {hwh_path}; PYNQ needs .bit and .hwh with the same basename'

print(f'Loading overlay: {bit_path}')
overlay = Overlay(str(bit_path))
print('Overlay loaded')
print('IP dictionary:')
for name, info in overlay.ip_dict.items():
    addr = info.get('phys_addr')
    size = info.get('addr_range')
    if addr is not None:
        print(f'  {name}: base=0x{addr:08x}, range=0x{size:x}')

print('Memory dictionary:')
for name, info in getattr(overlay, 'mem_dict', {}).items():
    addr = info.get('phys_addr')
    size = info.get('addr_range')
    if addr is not None:
        print(f'  {name}: base=0x{addr:08x}, range=0x{size:x}')

In [ ]:
def find_address(name_contains):
    matches = []
    dicts = [overlay.ip_dict, getattr(overlay, 'mem_dict', {})]
    for d in dicts:
        for name, info in d.items():
            if name_contains.lower() in name.lower() and 'phys_addr' in info:
                matches.append((name, info['phys_addr'], info.get('addr_range', 0)))
    if len(matches) != 1:
        raise RuntimeError(f'Expected one match for {name_contains!r}, got {matches}')
    return matches[0]


def find_any_address(candidates):
    errors = []
    for candidate in candidates:
        try:
            return find_address(candidate)
        except RuntimeError as exc:
            errors.append(str(exc))
    raise RuntimeError('\n'.join(errors))


ctrl_name, CTRL_BASE, CTRL_SIZE = find_any_address(['pimsr_axi_ip'])

try:
    lr_name, LR_BASE, BRAM_SIZE_LR = find_any_address(['lr_bram_BRAM_PORTA', 'lr_bram'])
except RuntimeError:
    lr_name, LR_BASE, BRAM_SIZE_LR = 'manual_lr_bram_from_address_editor', 0x42000000, 0x2000
    print('WARNING: LR BRAM address not found in .hwh by name; using manual 0x42000000')

try:
    hr_name, HR_BASE, BRAM_SIZE_HR = find_any_address(['hr_bram_BRAM_PORTA', 'hr_bram'])
except RuntimeError:
    hr_name, HR_BASE, BRAM_SIZE_HR = 'manual_hr_bram_from_address_editor', 0x40000000, 0x2000
    print('WARNING: HR BRAM address not found in .hwh by name; using manual 0x40000000')

print(f'Using CTRL: {ctrl_name} @ 0x{CTRL_BASE:08x}, size=0x{CTRL_SIZE:x}')
print(f'Using LR  : {lr_name} @ 0x{LR_BASE:08x}, size=0x{BRAM_SIZE_LR:x}')
print(f'Using HR  : {hr_name} @ 0x{HR_BASE:08x}, size=0x{BRAM_SIZE_HR:x}')

ctrl = MMIO(CTRL_BASE, CTRL_SIZE)
lr = MMIO(LR_BASE, BRAM_SIZE_LR)
hr = MMIO(HR_BASE, BRAM_SIZE_HR)

print('Initial status:', hex(ctrl.read(REG_CONTROL)))
print('Width register:', ctrl.read(REG_WIDTH))
print('Height register:', ctrl.read(REG_HEIGHT))

In [ ]:
print('Writing 4x4 LR image')
write_image_to_bram(lr, LR_4X4)
clear_bram(hr, IMG_W * IMG_H * SCALE * SCALE)

ctrl.write(REG_WIDTH, IMG_W)
ctrl.write(REG_HEIGHT, IMG_H)

print('Starting PIMSR IP')
ctrl.write(REG_CONTROL, 1)
status = wait_done(ctrl, timeout_s=1.0)
print('Done status:', hex(status))

In [ ]:
output = read_image_from_bram(hr, IMG_H * SCALE, IMG_W * SCALE)
print('HR output:')
print(output)

diff = output.astype(np.int16) - HR_8X8_GOLDEN.astype(np.int16)
mismatch = np.argwhere(diff != 0)
if mismatch.size == 0:
    print('PASS: hardware output matches RTL golden')
else:
    print(f'FAIL: {len(mismatch)} mismatched pixels')
    for y, x in mismatch[:16]:
        print(f'  y={y} x={x}: got=0x{output[y, x]:02x}, expected=0x{HR_8X8_GOLDEN[y, x]:02x}, diff={diff[y, x]}')